# Build COGs and STAC Catalog for ESF AGB Data

This notebook converts the ESF Landsat ensemble AGB (aboveground biomass) data from
per-year VRTs (each mosaicking ~432 GeoTIFF tiles on S3) into:

1. **One COG per year** — uploaded to publicly accessible OSN storage
2. **A static STAC catalog** — so tools can discover and load the data as a datacube

### Data summary
| | |
|---|---|
| Time steps | 34 annual (1990–2023) |
| Pixels per year | 24,000 × 18,000 (30 m resolution) |
| Size per year | ~1.7 GB uncompressed, ~650 MB as COG |
| CRS | EPSG:5070 (NAD83 / Conus Albers) |
| Source | `s3://cafri-share/landsat_ensemble_agb_2.0.0_*` |
| Output | `https://usgs.osn.mghpcc.org/esip/rsignell/esf/` |

## Setup

In [ ]:
import json
import os
import tempfile
from datetime import datetime, timezone
from pathlib import Path

import boto3
import pystac
import rasterio
from osgeo import gdal
from pyproj import Transformer
from shapely.geometry import box, mapping

In [ ]:
# ---------- Configuration ----------

VRT_DIR = Path('vrts')
YEARS = range(1990, 2024)  # all 34 years

# AWS profiles
READ_PROFILE = 'esf'           # for reading source tiles on cafri-share
WRITE_PROFILE = 'osn-esip'     # for writing to OSN

# OSN output paths
OSN_ENDPOINT = 'https://usgs.osn.mghpcc.org'
OSN_BUCKET = 'esip'
OSN_PREFIX = 'rsignell/esf-agb'    # COGs and STAC go here
PUBLIC_BASE_URL = f'{OSN_ENDPOINT}/{OSN_BUCKET}/{OSN_PREFIX}'

# Local scratch directory for COGs (one at a time, then uploaded)
LOCAL_COG_DIR = Path('cogs')
LOCAL_COG_DIR.mkdir(exist_ok=True)

# Delete local COG after upload to save disk space
CLEANUP_LOCAL = True

print(f'Public URL base: {PUBLIC_BASE_URL}/')

In [ ]:
# Configure GDAL to read /vsis3/ paths from the source bucket (cafri-share)
read_session = boto3.Session(profile_name=READ_PROFILE)
read_creds = read_session.get_credentials().get_frozen_credentials()

os.environ['AWS_ACCESS_KEY_ID'] = read_creds.access_key
os.environ['AWS_SECRET_ACCESS_KEY'] = read_creds.secret_key
if read_creds.token:
    os.environ['AWS_SESSION_TOKEN'] = read_creds.token
os.environ['AWS_DEFAULT_REGION'] = 'us-east-1'

gdal.SetConfigOption('GDAL_DISABLE_READDIR_ON_OPEN', 'EMPTY_DIR')
gdal.SetConfigOption('AWS_VIRTUAL_HOSTING', 'YES')

# Required for S3-compatible stores like OSN (Ceph-based)
os.environ['AWS_REQUEST_CHECKSUM_CALCULATION'] = 'WHEN_REQUIRED'
os.environ['AWS_RESPONSE_CHECKSUM_VALIDATION'] = 'WHEN_REQUIRED'

# fsspec filesystem for writing to OSN
import fsspec
osn_fs = fsspec.filesystem(
    's3',
    profile=WRITE_PROFILE,
    client_kwargs={'endpoint_url': OSN_ENDPOINT},
)

print('GDAL configured for cafri-share reads')
print('OSN fsspec filesystem configured for writes')

In [ ]:
for f in osn_fs.ls('esip/rsignell/esf/'):
    print(f,osn_fs.size(f)/1e9)

## Step 1 — Create COGs and upload to OSN

For each year:
1. `gdal.Translate` the VRT → local COG (DEFLATE compressed, 512×512 tiles, internal overviews)
2. Upload the COG to OSN
3. Optionally delete the local copy

Each COG is ~650 MB and takes several minutes to create (reads ~432 tiles from S3).

In [ ]:
from rio_cogeo.cogeo import cog_translate
from rio_cogeo.profiles import cog_profiles
from rasterio.crs import CRS


def create_cog(vrt_path, cog_path):
    """Convert a VRT to a Cloud Optimized GeoTIFF with proper EPSG:5070 CRS."""
    # Remove existing file to avoid COG layout conflict
    if cog_path.exists():
        cog_path.unlink()

    output_profile = cog_profiles.get("deflate")
    cog_translate(
        str(vrt_path),
        str(cog_path),
        output_profile,
        overview_resampling="average",
        use_cog_driver=True,
    )
    # The source tiles have a custom WKT (Albers + WGS84 datum) that doesn't
    # match EPSG:5070 (Albers + NAD83 datum). The projection parameters are
    # identical, so we can safely reassign to proper EPSG:5070 without
    # reprojecting any pixels. This fixes odc-stac / WarpedVRT NaN issues.
    with rasterio.open(str(cog_path), 'r+',
                       IGNORE_COG_LAYOUT_BREAK='YES') as dst:
        dst.crs = CRS.from_epsg(5070)
    return cog_path


def upload_to_osn(local_path, s3_key):
    """Upload a file to OSN using fsspec."""
    osn_fs.put(str(local_path), f'{OSN_BUCKET}/{s3_key}')

In [ ]:
%%time
import time

for year in YEARS:
    vrt_path = VRT_DIR / f'agb_{year}.vrt'
    cog_path = LOCAL_COG_DIR / f'agb_{year}_cog.tif'
    s3_key = f'{OSN_PREFIX}/agb_{year}_cog.tif'

    t0 = time.time()

    # Create COG (overwrites any existing local file)
    print(f'  {year}: creating COG...', end='', flush=True)
    create_cog(vrt_path, cog_path)
    size_mb = cog_path.stat().st_size / 1e6
    print(f' {size_mb:.0f} MB', end='', flush=True)

    # Upload (overwrites any existing file on OSN)
    print(f' → uploading...', end='', flush=True)
    upload_to_osn(cog_path, s3_key)

    elapsed = time.time() - t0
    print(f' done ({elapsed:.0f}s)')

    if CLEANUP_LOCAL and cog_path.exists():
        cog_path.unlink()

print('\nAll COGs uploaded.')

In [ ]:
# Verify the COG is accessible and has proper EPSG:5070 CRS
test_url = f'{PUBLIC_BASE_URL}/agb_1992_cog.tif'
print(test_url)
with rasterio.open(test_url) as src:
    print(f'URL:        {test_url}')
    print(f'Size:       {src.width} x {src.height}')
    print(f'CRS:        {src.crs}')
    print(f'Bounds:     {src.bounds}')
    print(f'Block size: {src.block_shapes}')
    print(f'Overviews:  {src.overviews(1)}')

## Step 2 — Build a STAC catalog

Create a static [STAC](https://stacspec.org/) catalog with:
- One **Collection** for the AGB dataset
- One **Item** per year, each pointing to its public COG URL

The catalog JSON files are uploaded alongside the COGs on OSN.

In [ ]:
# Get spatial extent and projection info from one COG (all years share the same grid)
sample_url = f'{PUBLIC_BASE_URL}/agb_1992_cog.tif'

with rasterio.open(sample_url) as src:
    native_bounds = src.bounds  # in EPSG:5070
    crs = src.crs
    native_transform = list(src.transform)[:6]  # affine coefficients
    native_shape = [src.height, src.width]       # [rows, cols]

# Transform bounds to EPSG:4326 for STAC
transformer = Transformer.from_crs(crs, 'EPSG:4326', always_xy=True)
west, south = transformer.transform(native_bounds.left, native_bounds.bottom)
east, north = transformer.transform(native_bounds.right, native_bounds.top)
bbox_4326 = [west, south, east, north]
geometry_4326 = mapping(box(*bbox_4326))

# Native CRS bbox for proj extension
proj_bbox = [native_bounds.left, native_bounds.bottom, native_bounds.right, native_bounds.top]

print(f'Native bounds (EPSG:5070): {native_bounds}')
print(f'Native shape (rows, cols): {native_shape}')
print(f'Native transform:          {native_transform}')
print(f'STAC bbox (EPSG:4326):     {[round(c, 4) for c in bbox_4326]}')

In [ ]:
# Create the STAC Collection
spatial_extent = pystac.SpatialExtent(bboxes=[bbox_4326])
temporal_extent = pystac.TemporalExtent(
    intervals=[[datetime(1990, 1, 1, tzinfo=timezone.utc),
                datetime(2023, 1, 1, tzinfo=timezone.utc)]]
)

collection = pystac.Collection(
    id='esf-agb',
    title='ESF Landsat Ensemble Aboveground Biomass',
    description=(
        'Annual aboveground biomass (AGB) carbon estimates (Mg/ha) for the '
        'eastern United States, derived from Landsat ensemble models. '
        '30 m resolution, EPSG:5070. Version 2.0.0.'
    ),
    license='proprietary',
    extent=pystac.Extent(spatial=spatial_extent, temporal=temporal_extent),
    providers=[
        pystac.Provider(
            name='ESF / Frontier Geospatial',
            roles=[pystac.ProviderRole.PRODUCER],
        ),
    ],
)

# Add items — one per year, with full projection extension metadata
for year in YEARS:
    cog_url = f'{PUBLIC_BASE_URL}/agb_{year}_cog.tif'

    item = pystac.Item(
        id=f'agb-{year}',
        geometry=geometry_4326,
        bbox=bbox_4326,
        datetime=datetime(year, 1, 1, tzinfo=timezone.utc),
        properties={
            'proj:epsg': 5070,
            'proj:shape': native_shape,
            'proj:transform': native_transform,
            'proj:bbox': proj_bbox,
        },
    )
    item.add_asset(
        'data',
        pystac.Asset(
            href=cog_url,
            media_type=pystac.MediaType.COG,
            roles=['data'],
            title=f'AGB {year}',
        ),
    )
    collection.add_item(item)

print(f'Collection has {len(list(collection.get_items()))} items')

In [ ]:
# Create a root catalog and add the collection
catalog = pystac.Catalog(
    id='esf-openfrontier',
    title='ESF OpenFrontier',
    description='Geospatial datasets from the ESF / Frontier Geospatial collaboration.',
)
catalog.add_child(collection)

# Set all hrefs relative to the public base URL
catalog.normalize_hrefs(f'{PUBLIC_BASE_URL}/stac')

# Validate
catalog.validate_all()
print('STAC catalog validates OK')

In [ ]:
# Save locally, then upload JSON files to OSN
local_stac_dir = Path('stac')
catalog.save(catalog_type=pystac.CatalogType.SELF_CONTAINED, dest_href=str(local_stac_dir))

# Upload all JSON files
for json_path in local_stac_dir.rglob('*.json'):
    rel_path = json_path.relative_to(local_stac_dir)
    s3_key = f'{OSN_PREFIX}/stac/{rel_path}'
    osn_fs.put(str(json_path), f'{OSN_BUCKET}/{s3_key}')
    print(f'  uploaded {s3_key}')

stac_url = f'{PUBLIC_BASE_URL}/stac/catalog.json'
print(f'\nSTAC catalog URL: {stac_url}')

## Step 3 — Verify

Read the STAC catalog back from its public URL and load data as an xarray datacube.

In [ ]:
# Read the catalog back from the public URL
cat = pystac.Catalog.from_file(f'{PUBLIC_BASE_URL}/stac/catalog.json')

col = list(cat.get_children())[0]
items = list(col.get_all_items())

print(f'Catalog: {cat.title}')
print(f'Collection: {col.title}')
print(f'Items: {len(items)}')
print(f'First item: {items[0].id} → {items[0].assets["data"].href}')

In [ ]:
import odc.stac

# Now that the COG has proper EPSG:5070, odc-stac should work without CRS mismatch
ds = odc.stac.load(
    items,
    crs='EPSG:5070',
    resolution=30,
    chunks={'x': 2048, 'y': 2048, 'time': 1},
)
ds

In [ ]:
%%time
# Load a spatial subset via odc-stac
subset = ds['data'].isel(time=0, y=slice(1000, 4000), x=slice(12000, 16000)).load()

import numpy as np
print(f'Min: {np.nanmin(subset.values):.2f}, Max: {np.nanmax(subset.values):.2f}')
print(f'NaN: {np.isnan(subset.values).sum()} / {subset.size}')
subset.plot(figsize=(8, 6), cmap='YlOrBr', clim=(0, 300))

In [ ]:
import hvplot.xarray

subset.hvplot(x='x', y='y', rasterize=True, crs='EPSG:5070', tiles='OSM', 
              alpha=0.5, cmap='YlOrBr', title='Carbon in 1992 (Mg/ha)')

## How to use the data

### ArcGIS Pro
Open any individual COG directly by URL:
```
https://usgs.osn.mghpcc.org/esip/rsignell/esf/agb_2020_cog.tif
```

### Web visualization with titiler
View any COG in a titiler instance, e.g.:
```
https://titiler.xyz/cog/viewer?url=https://usgs.osn.mghpcc.org/esip/rsignell/esf/agb_2020_cog.tif
```

### Python / xarray
```python
import pystac
import xarray as xr
import hvplot.xarray

# Discover data via STAC
catalog = pystac.Catalog.from_file(
    'https://usgs.osn.mghpcc.org/esip/rsignell/esf/stac/catalog.json'
)
collection = list(catalog.get_children())[0]
items = {item.id: item for item in collection.get_all_items()}

# Open a single year
url = items['agb-2020'].assets['data'].href
ds = xr.open_dataset(url, engine='rasterio').squeeze('band', drop=True)

# Subset and visualize
da = ds['band_data'][1000:4000, 12000:16000].load()
da.hvplot(x='x', y='y', rasterize=True, crs='EPSG:5070', tiles='OSM',
          alpha=0.5, cmap='YlOrBr', title='Carbon in 2020 (Mg/ha)')
```